In [ ]:
#@title Setup -- avaliacao final (testes estatisticos + tabela comparativa)
from pathlib import Path
PROJECT = Path(r'D:\tcc_ibov_corpo_completo')
DIRS = {k: PROJECT / v for k, v in {
    'raw': 'data/raw', 'processed': 'data/processed',
    'models': 'models', 'reports': 'reports', 'utils': 'utils',
}.items()}

import sys; sys.path.insert(0, str(DIRS['utils']))
import importlib, tcc_utils; importlib.reload(tcc_utils)
from tcc_utils import make_target, walk_forward_splits

import numpy as np
import pandas as pd
from scipy import stats

print("Setup OK.")


In [ ]:
#@title Geometria dos folds: baseline (embargo=0) vs LSTM (embargo=21)
# Ambas as familias particionam o MESMO indice (features_preco.parquet, 1492
# pregoes), com os MESMOS train_end por fold -- a unica diferenca e o embargo
# de 21 dias (=lookback) antes do inicio do teste do LSTM. Isso faz o teste do
# LSTM ser um SUBCONJUNTO (deslocado) do teste do baseline em cada fold, o que
# permite comparar as duas familias na INTERSECCAO de dias, sem vazamento e
# sem inventar dados.

feat_preco = pd.read_parquet(DIRS['processed'] / 'features_preco.parquet')
y_all = feat_preco['y'].values
N_SPLITS, MIN_TRAIN, LOOKBACK = 5, 252, 21

splits_base = walk_forward_splits(feat_preco.index, n_splits=N_SPLITS,
                                   min_train=MIN_TRAIN, expanding=True, embargo=0)
splits_lstm = walk_forward_splits(feat_preco.index, n_splits=N_SPLITS,
                                   min_train=MIN_TRAIN, expanding=True, embargo=LOOKBACK)

print("Overlap baseline/LSTM por fold (dias de teste em comum):")
overlap_idx = {}   # fold -> (posicoes globais na interseccao, ordenadas)
for k, ((_, te_b), (_, te_l)) in enumerate(zip(splits_base, splits_lstm), 1):
    common = np.array(sorted(set(te_b) & set(te_l)))
    overlap_idx[k] = common
    print(f"  fold {k}: baseline {len(te_b)} | lstm {len(te_l)} | "
          f"interseccao {len(common)} dias "
          f"({feat_preco.index[common[0]].date()} -> {feat_preco.index[common[-1]].date()})")

assert all(len(v) > 0 for v in overlap_idx.values()), "fold sem interseccao -- revisar geometria"
print("\nOK -- interseccao nao-vazia em todos os folds.")


In [ ]:
#@title Carrega previsoes de todos os modelos (por fold, na ordem dos dias de teste)
# Random Walk nao tem .npz salvo (previsao trivial: 0 p/ regressao, sinal do
# retorno de hoje p/ direcao) -- reconstruida aqui a partir dos mesmos dados
# usados no notebook 04, célula do RW.
precos_estudo = feat_preco  # já é o período de estudo, 1492 dias
ret_hoje = precos_estudo['ret_lag0'].values  # retorno de hoje (conhecido em t)

def fold_positions(splits, k):
    return splits[k - 1][1]  # posições de teste do fold k (1-indexado)

modelos = {}  # nome -> {'family': ..., 'preds': {k: array}, 'y_true': {k: array}}

# --- Random Walk (family baseline, embargo=0) ---
preds_rw = {}
for k in range(1, N_SPLITS + 1):
    te = fold_positions(splits_base, k)
    preds_rw[k] = np.zeros(len(te))  # previsão de retorno = 0
modelos['RandomWalk'] = {'family': 'baseline', 'preds': preds_rw}

# --- ARIMA / ARIMAX (baseline) ---
d = np.load(DIRS['reports'] / 'preds_arima.npz')
modelos['ARIMA']  = {'family': 'baseline', 'preds': {k: d[f'arima_f{k}']  for k in range(1, 6)}}
modelos['ARIMAX'] = {'family': 'baseline', 'preds': {k: d[f'arimax_f{k}'] for k in range(1, 6)}}

# --- XGBoost (baseline; usa a saída do REGRESSOR p/ ser comparável em escala
#     de retorno com o LSTM -- não a probabilidade do classificador) ---
d = np.load(DIRS['reports'] / 'preds_xgboost.npz')
modelos['XGBoost'] = {'family': 'baseline', 'preds': {k: d[f'reg_f{k}'] for k in range(1, 6)}}

# --- LSTM (família lstm, embargo=21) ---
lstm_files = {
    'LSTM-preco':          'preds_lstm_preco.npz',
    'LSTM-BERTimbau-k8':   'preds_lstm_texto.npz',
    'LSTM-BERTimbau-k30':  'preds_lstm_texto_k30.npz',
    'LSTM-FinBERT-sent':   'preds_lstm_via2_sent.npz',
    'LSTM-FinBERT-emb-k8': 'preds_lstm_via1_finbert_k8.npz',
    'LSTM-FinBERT-emb-k30':'preds_lstm_via1_finbert_k30.npz',
    'LSTM-Copom':          'preds_lstm_via3_copom.npz',
}
for nome, arq in lstm_files.items():
    d = np.load(DIRS['reports'] / arq)
    modelos[nome] = {'family': 'lstm', 'preds': {k: d[f'lstm_f{k}'] for k in range(1, 6)}}

# y_true por fold, em cada família (mesma ordem de posições que as preds)
y_true_base = {k: y_all[fold_positions(splits_base, k)] for k in range(1, 6)}
y_true_lstm = {k: y_all[fold_positions(splits_lstm, k)] for k in range(1, 6)}

for nome, m in modelos.items():
    yt = y_true_base if m['family'] == 'baseline' else y_true_lstm
    for k in range(1, 6):
        assert len(m['preds'][k]) == len(yt[k]), f"{nome} fold {k}: shape mismatch"
    m['y_true'] = yt

print(f"{len(modelos)} modelos carregados: {list(modelos.keys())}")


In [ ]:
#@title Diebold-Mariano (erro quadratico) e Pesaran-Timmermann (direcao)
# DM: Diebold & Mariano (1995), "Comparing Predictive Accuracy", JBES.
# Correção de amostra pequena: Harvey, Leybourne & Newbold (1997), "Testing
# the equality of prediction mean squared errors", IJF -- multiplica a
# estatística por sqrt((n+1-2h+h(h-1)/n)/n) e usa t-Student(n-1) em vez de
# normal. h=1 (previsão 1 passo à frente, sem sobreposição de horizonte).
def diebold_mariano(y_true, pred1, pred2, h=1):
    e1 = y_true - pred1
    e2 = y_true - pred2
    d = e1**2 - e2**2                      # d>0 => modelo 1 pior (erro maior)
    n = len(d)
    dbar = d.mean()
    var_d = d.var(ddof=1)
    if var_d <= 1e-18:
        return 0.0, 1.0, dbar
    dm_stat = dbar / np.sqrt(var_d / n)
    hln = np.sqrt((n + 1 - 2*h + h*(h-1)/n) / n)
    dm_adj = dm_stat * hln
    p = 2 * (1 - stats.t.cdf(abs(dm_adj), df=n - 1))
    return dm_adj, p, dbar

# PT: Pesaran & Timmermann (1992), "A simple nonparametric test of predictive
# performance", JBES -- testa se a acurácia direcional excede o esperado sob
# independência entre sinal previsto e sinal realizado. Implementação idêntica
# à usada no notebook 04 (baselines), reaproveitada aqui p/ TODOS os modelos.
def pesaran_timmermann(y_true, y_pred):
    n = len(y_true)
    dy = (y_true > 0).astype(int); dp = (y_pred > 0).astype(int)
    py, pp = dy.mean(), dp.mean()
    p_hat = (dy == dp).mean()
    p_star = py*pp + (1-py)*(1-pp)
    var_hat = p_star*(1-p_star)/n
    var_star = (((2*py-1)**2)*pp*(1-pp)/n
                + ((2*pp-1)**2)*py*(1-py)/n
                + 4*py*pp*(1-py)*(1-pp)/n**2)
    denom = np.sqrt(max(var_hat - var_star, 1e-12))
    z = (p_hat - p_star) / denom
    return z, 1 - stats.norm.cdf(z)   # unilateral (H1: melhor que o acaso)

print("Funções de teste definidas.")


In [ ]:
#@title Pesaran-Timmermann pooled (todos os folds concatenados), por modelo
# Para o PT, a regra direcional do Random Walk e a PERSISTENCIA (sinal do
# retorno de hoje) -- a mesma usada nas colunas DA/F1/MCC da tabela do RW.
# A previsao de REGRESSAO do RW (constante zero) e degenerada num teste
# direcional (sign(0) nunca coincide com o sinal realizado) e nao testa
# nada; ela permanece sendo usada na regressao e no backtest financeiro
# (posicao 0 = nunca opera), com nota na legenda da tabela.
ret_hoje_all = feat_preco['ret_lag0'].values
persist_preds = {k: np.sign(ret_hoje_all[fold_positions(splits_base, k)])
                 for k in range(1, 6)}

pt_rows = []
for nome, m in modelos.items():
    preds_para_pt = persist_preds if nome == 'RandomWalk' else m['preds']
    y_cat = np.concatenate([m['y_true'][k] for k in range(1, 6)])
    p_cat = np.concatenate([preds_para_pt[k] for k in range(1, 6)])
    da = (np.sign(y_cat) == np.sign(p_cat)).mean()
    z, p = pesaran_timmermann(y_cat, p_cat)
    pt_rows.append({'modelo': nome, 'familia': m['family'], 'n_dias': len(y_cat),
                     'DA_pooled': da, 'PT_z': z, 'PT_p': p})

pt_df = pd.DataFrame(pt_rows).set_index('modelo')

from statsmodels.stats.multitest import multipletests
rej, p_holm, _, _ = multipletests(pt_df['PT_p'].values, alpha=0.05, method='holm')
pt_df['PT_p_holm'] = p_holm
pt_df['significativo_5pct'] = rej

print("=== Pesaran-Timmermann pooled (H1: acurácia direcional > acaso) ===")
print(pt_df.round(4).to_string())

pt_df.to_parquet(DIRS['reports'] / 'testes_pt_pooled.parquet')
print("\nSalvo: testes_pt_pooled.parquet")


In [ ]:
#@title DM pareado -- LSTM-preço vs cada variante de texto (mesmos folds, sem interseção)
# Todas as variantes LSTM compartilham EXATAMENTE os mesmos splits (embargo=21
# sobre o mesmo índice) -- não há necessidade de interseção aqui, os dias de
# teste já coincidem fold a fold.
texto_variantes = ['LSTM-BERTimbau-k8', 'LSTM-BERTimbau-k30', 'LSTM-FinBERT-sent',
                    'LSTM-FinBERT-emb-k8', 'LSTM-FinBERT-emb-k30', 'LSTM-Copom']

dm_rows = []
y_preco_cat = np.concatenate([modelos['LSTM-preco']['y_true'][k] for k in range(1, 6)])
p_preco_cat = np.concatenate([modelos['LSTM-preco']['preds'][k]  for k in range(1, 6)])

for nome in texto_variantes:
    y_txt_cat = np.concatenate([modelos[nome]['y_true'][k] for k in range(1, 6)])
    p_txt_cat = np.concatenate([modelos[nome]['preds'][k]  for k in range(1, 6)])
    assert np.allclose(y_preco_cat, y_txt_cat), f"{nome}: y_true diverge do LSTM-preço -- folds não batem"
    dm, p, dbar = diebold_mariano(y_preco_cat, p_preco_cat, p_txt_cat)
    dm_rows.append({'comparacao': f'LSTM-preço vs {nome}', 'familia': 'texto-vs-preço',
                     'DM_stat': dm, 'DM_p': p,
                     'dbar (>0: preço pior, texto ajuda)': dbar, 'n': len(y_preco_cat)})

dm_texto_df = pd.DataFrame(dm_rows)
print("=== DM: LSTM-preço vs variantes de texto (H0: mesma acurácia de previsão) ===")
print(dm_texto_df.round(5).to_string(index=False))


In [ ]:
#@title DM pareado -- XGBoost (melhor baseline tabular) vs cada variante LSTM (interseção)
# XGBoost (embargo=0) e LSTM (embargo=21) cobrem janelas de teste diferentes
# por fold -- comparamos SÓ na interseção (ver célula "Geometria dos folds"),
# preservando pareamento dia-a-dia exigido pelo DM test.

def preds_na_interseccao(modelo, k, splits, common_positions):
    """Extrai, do array de previsões do fold k (na ordem de teste do próprio
    split do modelo), só os valores cujas posições globais estão na interseção."""
    te = splits[k - 1][1]
    pos_to_idx = {pos: i for i, pos in enumerate(te)}
    idx = [pos_to_idx[p] for p in common_positions]
    return modelo['preds'][k][idx]

lstm_variantes_todas = ['LSTM-preco'] + texto_variantes
dm_xgb_rows = []
for nome in lstm_variantes_todas:
    y_int, p_xgb_int, p_lstm_int = [], [], []
    for k in range(1, 6):
        common = overlap_idx[k]
        y_int.append(y_all[common])
        p_xgb_int.append(preds_na_interseccao(modelos['XGBoost'], k, splits_base, common))
        p_lstm_int.append(preds_na_interseccao(modelos[nome], k, splits_lstm, common))
    y_int = np.concatenate(y_int); p_xgb_int = np.concatenate(p_xgb_int); p_lstm_int = np.concatenate(p_lstm_int)

    dm, p, dbar = diebold_mariano(y_int, p_xgb_int, p_lstm_int)
    dm_xgb_rows.append({'comparacao': f'XGBoost vs {nome}', 'familia': 'baseline-vs-lstm',
                         'DM_stat': dm, 'DM_p': p,
                         'dbar (>0: XGBoost pior, LSTM ajuda)': dbar, 'n': len(y_int)})

dm_xgb_df = pd.DataFrame(dm_xgb_rows)
print("=== DM: XGBoost vs variantes LSTM (na interseção de dias de teste) ===")
print(dm_xgb_df.round(5).to_string(index=False))


In [ ]:
#@title Correção de múltiplas comparações (Holm-Bonferroni) e tabela de testes
dm_all = pd.concat([dm_texto_df, dm_xgb_df], ignore_index=True)
rej, p_holm, _, _ = multipletests(dm_all['DM_p'].values, alpha=0.05, method='holm')
dm_all['DM_p_holm'] = p_holm
dm_all['significativo_5pct'] = rej

print("=== DM -- todas as comparações, com correção de Holm (família de "
      f"{len(dm_all)} testes) ===")
print(dm_all[['comparacao', 'DM_stat', 'DM_p', 'DM_p_holm', 'significativo_5pct']]
      .round(5).to_string(index=False))

n_sig = dm_all['significativo_5pct'].sum()
print(f"\n{n_sig}/{len(dm_all)} comparações significativas a 5% após correção de Holm.")

dm_all.to_parquet(DIRS['reports'] / 'testes_dm_pareados.parquet')
print("\nSalvo: testes_dm_pareados.parquet")


In [ ]:
#@title Tabela comparativa consolidada -- todos os modelos
FONTES = {
    'RandomWalk':            ('baseline_random_walk.parquet', None),
    'ARIMA':                 ('baseline_arima.parquet', None),
    'ARIMAX':                ('baseline_arimax.parquet', None),
    'XGBoost':                ('baseline_xgboost_full.parquet', 'seed'),
    'LSTM-preco':             ('lstm_preco_full.parquet', 'seed'),
    'LSTM-BERTimbau-k8':      ('lstm_texto_full.parquet', 'seed'),
    'LSTM-BERTimbau-k30':     ('lstm_texto_k30_full.parquet', 'seed'),
    'LSTM-FinBERT-sent':      ('lstm_via2_sent_full.parquet', 'seed'),
    'LSTM-FinBERT-emb-k8':    ('lstm_via1_finbert_k8_full.parquet', 'seed'),
    'LSTM-FinBERT-emb-k30':   ('lstm_via1_finbert_k30_full.parquet', 'seed'),
    'LSTM-Copom':             ('lstm_via3_copom_full.parquet', 'seed'),
}
RENAME_RW = {'MAE_rw0':'MAE','RMSE_rw0':'RMSE','DA_persist':'DA','F1_persist':'F1',
             'MCC_persist':'MCC','Brier_clim':'Brier'}

linhas = []
for nome, (arq, seed_col) in FONTES.items():
    df = pd.read_parquet(DIRS['reports'] / arq)
    if nome == 'RandomWalk':
        df = df.rename(columns=RENAME_RW)
    metric_cols = [c for c in ['MAE','RMSE','DA','F1','MCC','Brier'] if c in df.columns]

    if seed_col is not None:
        # media sobre sementes por fold, depois media entre folds (resumo);
        # desvio combinando a variancia entre sementes de cada fold (pooled)
        resumo = df.groupby('fold')[metric_cols].mean().mean()
        desvio = ((df.groupby('fold')[metric_cols].std() ** 2).mean()) ** 0.5
    else:
        resumo = df.set_index('fold')[metric_cols].mean() if 'fold' in df.columns else df[metric_cols].mean()
        desvio = pd.Series(0.0, index=metric_cols)  # deterministico -- sem variacao entre execucoes

    row = {'modelo': nome}
    for m in metric_cols:
        row[m] = resumo[m]
        row[f'{m}_sd'] = desvio[m]
    linhas.append(row)

tabela = pd.DataFrame(linhas).set_index('modelo')

# Junta PT pooled e flag de significancia DM (quando o modelo aparece como
# "desafiante" testado contra XGBoost ou contra LSTM-preco)
tabela = tabela.join(pt_df[['DA_pooled', 'PT_z', 'PT_p', 'PT_p_holm']])

sig_vs_xgb = dm_all[dm_all['comparacao'].str.startswith('XGBoost vs ')].copy()
sig_vs_xgb['modelo'] = sig_vs_xgb['comparacao'].str.replace('XGBoost vs ', '', regex=False)
sig_vs_xgb = sig_vs_xgb.set_index('modelo')['significativo_5pct']
tabela['sig_dm_vs_XGBoost'] = sig_vs_xgb.reindex(tabela.index)

sig_vs_preco = dm_all[dm_all['comparacao'].str.startswith('LSTM-preço vs ')].copy()
sig_vs_preco['modelo'] = sig_vs_preco['comparacao'].str.replace('LSTM-preço vs ', '', regex=False)
sig_vs_preco = sig_vs_preco.set_index('modelo')['significativo_5pct']
tabela['sig_dm_vs_LSTMpreco'] = sig_vs_preco.reindex(tabela.index)

cols_show = ['MAE','RMSE','DA','F1','MCC','MCC_sd','Brier','PT_p_holm',
             'sig_dm_vs_XGBoost','sig_dm_vs_LSTMpreco']
print("=== Tabela comparativa final (arredondado) ===")
print(tabela[cols_show].round(4).to_string())

tabela.to_parquet(DIRS['reports'] / 'tabela_comparativa_final.parquet')
tabela[cols_show].round(4).to_csv(DIRS['reports'] / 'tabela_comparativa_final.csv')
print("\nSalvos: tabela_comparativa_final.parquet, tabela_comparativa_final.csv")


In [ ]:
#@title Metricas financeiras -- Sharpe, max drawdown, com e sem custo de transacao
# Estrategia mecanica: posicao_t = sinal(previsao_t) em {-1,0,+1} (long/short,
# sem zona morta). Retorno liquido do dia: r_t = posicao_t * y_t - custo_t,
# custo_t = TC * |posicao_t - posicao_{t-1}| (turnover; TC em bps por unidade
# de posicao trocada -- cobre fechar+abrir numa reversao completa).
# y_t ja e log-retorno de D+1 (make_target); tratamos posicao*y_t como
# aproximacao padrao do retorno log da estrategia (uso comum em ML financeiro
# para custos pequenos). Cada modelo e avaliado no seu PROPRIO periodo de
# teste (folds da sua familia -- baseline embargo=0 ou LSTM embargo=21),
# concatenado em ordem cronologica (folds sao disjuntos e sequenciais).
TC_BPS = 5
TC = TC_BPS / 10000

def backtest(y_true, pred, tc):
    pos = np.sign(pred)
    pos_prev = np.concatenate([[0.0], pos[:-1]])
    turnover = np.abs(pos - pos_prev)
    r_net = pos * y_true - tc * turnover
    wealth = np.exp(np.cumsum(r_net))
    running_max = np.maximum.accumulate(wealth)
    drawdown = wealth / running_max - 1
    max_dd = drawdown.min()
    sharpe = (r_net.mean() / r_net.std() * np.sqrt(252)) if r_net.std() > 1e-12 else 0.0
    n_trades = int((turnover > 0).sum())
    return {'Sharpe': sharpe, 'MaxDrawdown': max_dd, 'RetAcum': wealth[-1] - 1,
            'N_trades': n_trades, 'Turnover_medio': turnover.mean()}

fin_rows = []
for nome, m in modelos.items():
    y_cat = np.concatenate([m['y_true'][k] for k in range(1, 6)])
    p_cat = np.concatenate([m['preds'][k]  for k in range(1, 6)])

    res_0bps = backtest(y_cat, p_cat, tc=0.0)
    res_tc   = backtest(y_cat, p_cat, tc=TC)

    fin_rows.append({
        'modelo': nome,
        'Sharpe_0bps': res_0bps['Sharpe'], 'MaxDD_0bps': res_0bps['MaxDrawdown'],
        'RetAcum_0bps': res_0bps['RetAcum'],
        f'Sharpe_{TC_BPS}bps': res_tc['Sharpe'], f'MaxDD_{TC_BPS}bps': res_tc['MaxDrawdown'],
        f'RetAcum_{TC_BPS}bps': res_tc['RetAcum'],
        'N_trades': res_tc['N_trades'], 'Turnover_medio': res_tc['Turnover_medio'],
    })

fin_df = pd.DataFrame(fin_rows).set_index('modelo')

# Buy-and-hold como referencia (posicao sempre +1, sem custo -- benchmark
# passivo, nao e uma "previsao", so contextualiza a escala dos retornos)
bh_y = np.concatenate([modelos['XGBoost']['y_true'][k] for k in range(1, 6)])  # mesmo periodo baseline
bh_wealth = np.exp(np.cumsum(bh_y))
bh_dd = (bh_wealth / np.maximum.accumulate(bh_wealth) - 1).min()
bh_sharpe = bh_y.mean() / bh_y.std() * np.sqrt(252)
print(f"Buy-and-hold (referencia, período baseline): Sharpe={bh_sharpe:.3f} | "
      f"MaxDD={bh_dd:.2%} | RetAcum={bh_wealth[-1]-1:.2%}\n")

print(f"=== Métricas financeiras por modelo (long/short, custo 0 vs {TC_BPS} bps/trade) ===")
print(fin_df.round(4).to_string())

fin_df.to_parquet(DIRS['reports'] / 'metricas_financeiras.parquet')
fin_df.round(4).to_csv(DIRS['reports'] / 'metricas_financeiras.csv')
print("\nSalvos: metricas_financeiras.parquet, metricas_financeiras.csv")
